# 📘 Module 2.1 — Convolution Fundamentals

**Goal:** Understand convolution operations — the building blocks of all vision models in ADAS.

## Why CNNs for ADAS?
CNNs are the **backbone of computer vision** in autonomous driving:
- 🚗 **Object Detection:** Detecting cars, pedestrians, cyclists
- 🛣️ **Lane Detection:** Finding lane markings
- 🚦 **Traffic Sign Recognition:** Reading speed limits, stop signs
- 🏗️ **Semantic Segmentation:** Understanding drivable area vs. sidewalk

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

## 1. What is Convolution?

A **convolution** slides a small filter (kernel) over an image and computes dot products at each position.

```
Input Image (5×5)          Kernel (3×3)           Output (3×3)
┌───┬───┬───┬───┬───┐    ┌───┬───┬───┐    ┌───┬───┬───┐
│ 1 │ 0 │ 1 │ 0 │ 1 │    │ 1 │ 0 │ 1 │    │   │   │   │
├───┼───┼───┼───┼───┤    ├───┼───┼───┤    ├───┼───┼───┤
│ 0 │ 1 │ 0 │ 1 │ 0 │  * │ 0 │ 1 │ 0 │  = │   │   │   │
├───┼───┼───┼───┼───┤    ├───┼───┼───┤    ├───┼───┼───┤
│ 1 │ 0 │ 1 │ 0 │ 1 │    │ 1 │ 0 │ 1 │    │   │   │   │
├───┼───┼───┼───┼───┤    └───┴───┴───┘    └───┴───┴───┘
│ 0 │ 1 │ 0 │ 1 │ 0 │
├───┼───┼───┼───┼───┤
│ 1 │ 0 │ 1 │ 0 │ 1 │
└───┴───┴───┴───┴───┘
```

**Key Insight:** The kernel learns to detect **specific patterns** (edges, textures, shapes).

In [ ]:
# --- Manual Convolution Example ---
# Let's see convolution step by step

# Create a simple 5x5 "image"
image = torch.tensor([
    [0., 0., 0., 0., 0.],
    [0., 0., 1., 0., 0.],
    [0., 1., 1., 1., 0.],
    [0., 0., 1., 0., 0.],
    [0., 0., 0., 0., 0.]
])

# Edge detection kernel (Laplacian)
kernel = torch.tensor([
    [ 0., -1.,  0.],
    [-1.,  4., -1.],
    [ 0., -1.,  0.]
])

# PyTorch conv2d expects: (batch, channels, H, W)
image_4d = image.unsqueeze(0).unsqueeze(0)   # (1, 1, 5, 5)
kernel_4d = kernel.unsqueeze(0).unsqueeze(0)  # (1, 1, 3, 3)

output = F.conv2d(image_4d, kernel_4d)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image, cmap='gray')
axes[0].set_title('Input Image')
axes[1].imshow(kernel, cmap='RdBu_r')
axes[1].set_title('Edge Detection Kernel')
axes[2].imshow(output.squeeze(), cmap='RdBu_r')
axes[2].set_title('Output (Edges Detected!)')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Input shape: {image.shape} → Output shape: {output.squeeze().shape}")

In [ ]:
# --- Classic Image Filters as Kernels ---
# These are the kinds of patterns CNNs learn automatically!

# Create a synthetic "road scene" image
np.random.seed(42)
scene = np.zeros((64, 64))
# Add a horizontal line (lane marking)
scene[30:32, 10:54] = 1.0
# Add a vertical line (pole)
scene[10:50, 40:42] = 1.0
# Add a square (vehicle)
scene[15:25, 15:25] = 0.8
# Add some noise
scene += np.random.randn(64, 64) * 0.1

scene_tensor = torch.tensor(scene, dtype=torch.float32).unsqueeze(0).unsqueeze(0)

# Define classic kernels
kernels = {
    'Horizontal Edge': torch.tensor([[-1., -1., -1.], [0., 0., 0.], [1., 1., 1.]]),
    'Vertical Edge': torch.tensor([[-1., 0., 1.], [-1., 0., 1.], [-1., 0., 1.]]),
    'Sharpen': torch.tensor([[0., -1., 0.], [-1., 5., -1.], [0., -1., 0.]]),
    'Blur': torch.ones(3, 3) / 9.0,
}

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
axes[0].imshow(scene, cmap='gray')
axes[0].set_title('Original Scene')

for ax, (name, k) in zip(axes[1:], kernels.items()):
    k_4d = k.unsqueeze(0).unsqueeze(0)
    result = F.conv2d(scene_tensor, k_4d, padding=1)
    ax.imshow(result.squeeze().numpy(), cmap='gray')
    ax.set_title(name)

for ax in axes:
    ax.axis('off')
plt.suptitle('Effect of Different Convolution Kernels', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Key Convolution Parameters

| Parameter | What it does | Typical values |
|-----------|-------------|----------------|
| **Kernel Size** | Size of the sliding window | 3×3, 5×5, 7×7 |
| **Stride** | How many pixels to skip | 1 (default), 2 (downsamples) |
| **Padding** | Zeros added around input | 0, 1, 'same' |
| **Channels** | Input and output feature maps | 3→64, 64→128, etc. |

In [ ]:
# --- Padding and Stride effects ---

x = torch.randn(1, 1, 8, 8)  # 8x8 input

# No padding, stride=1
conv1 = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=0)
print(f"No padding, stride=1: {x.shape} → {conv1(x).shape}")

# Same padding (output = input size)
conv2 = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1)
print(f"Padding=1, stride=1: {x.shape} → {conv2(x).shape}")

# Stride=2 (downsamples by 2x)
conv3 = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1)
print(f"Padding=1, stride=2: {x.shape} → {conv3(x).shape}")

# Multiple output channels
conv4 = nn.Conv2d(3, 64, kernel_size=3, padding=1)  # RGB → 64 features
rgb = torch.randn(1, 3, 224, 224)
print(f"\nRGB → 64 features: {rgb.shape} → {conv4(rgb).shape}")

## 3. Pooling Layers

Pooling **reduces spatial dimensions** while keeping important features.

```
Max Pooling (2×2):
┌───┬───┬───┬───┐         ┌───┬───┐
│ 1 │ 3 │ 2 │ 4 │         │ 3 │ 4 │
├───┼───┼───┼───┤   →     ├───┼───┤
│ 5 │ 2 │ 8 │ 1 │         │ 5 │ 8 │
├───┼───┼───┼───┤         └───┴───┘
│ 4 │ 1 │ 3 │ 7 │
├───┼───┼───┼───┤
│ 2 │ 6 │ 5 │ 9 │
└───┴───┴───┴───┘
```

In [ ]:
# --- Pooling Operations ---
x = torch.randn(1, 64, 32, 32)  # 64 feature maps, 32x32

# Max pooling (most common)
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
print(f"MaxPool 2x2: {x.shape} → {maxpool(x).shape}")

# Average pooling
avgpool = nn.AvgPool2d(kernel_size=2, stride=2)
print(f"AvgPool 2x2: {x.shape} → {avgpool(x).shape}")

# Adaptive average pooling (used in modern architectures)
adaptive = nn.AdaptiveAvgPool2d((1, 1))  # Output is always 1x1
print(f"AdaptiveAvgPool: {x.shape} → {adaptive(x).shape}")

## 4. Batch Normalization

**Batch Normalization** normalizes layer inputs, making training faster and more stable. Used in almost every modern CNN.

In [ ]:
# BatchNorm normalizes across the batch for each channel
bn = nn.BatchNorm2d(64)  # 64 channels
x = torch.randn(16, 64, 32, 32)  # batch of 16

out = bn(x)
print(f"Before BN — mean: {x[:,0].mean():.4f}, std: {x[:,0].std():.4f}")
print(f"After BN  — mean: {out[:,0].mean():.4f}, std: {out[:,0].std():.4f}")

## 5. Building a Complete CNN

Let's build a CNN for classifying CIFAR-10 images (tiny 32×32 images of 10 categories).

```
Input (3,32,32)
    │
    ▼ Conv(3→32) + BN + ReLU + Pool
Feature Maps (32,16,16)
    │
    ▼ Conv(32→64) + BN + ReLU + Pool
Feature Maps (64,8,8)
    │
    ▼ Conv(64→128) + BN + ReLU + Pool
Feature Maps (128,4,4)
    │
    ▼ Flatten + FC(2048→256) + FC(256→10)
Output (10 classes)
```

In [ ]:
class SimpleCNN(nn.Module):
    """A simple CNN for CIFAR-10 classification."""
    
    def __init__(self, num_classes=10):
        super(SimpleCNN, self).__init__()
        
        # Convolutional blocks
        self.features = nn.Sequential(
            # Block 1: 3 → 32 channels
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 32x32 → 16x16
            
            # Block 2: 32 → 64 channels
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 16x16 → 8x8
            
            # Block 3: 64 → 128 channels
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),  # 8x8 → 4x4
        )
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Create and inspect
model = SimpleCNN()
print(model)

# Test with random input
test_input = torch.randn(4, 3, 32, 32)  # 4 RGB images
test_output = model(test_input)
print(f"\nInput: {test_input.shape} → Output: {test_output.shape}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# --- Visualize feature map sizes through the network ---
x = torch.randn(1, 3, 32, 32)
print(f"Input: {x.shape}")

# Pass through each block manually to see shapes
for i, layer in enumerate(model.features):
    x = layer(x)
    if isinstance(layer, (nn.Conv2d, nn.MaxPool2d)):
        print(f"After {layer.__class__.__name__}: {x.shape}")

---
## ✅ Key Takeaways

1. **Convolution** = sliding a learned filter over an image to detect patterns
2. **Kernels** learn to detect edges → textures → parts → objects (hierarchically)
3. **Stride** controls downsampling; **Padding** controls output size
4. **Pooling** reduces spatial dimensions while keeping important features
5. **Batch Normalization** makes training faster and more stable
6. **CNN pattern:** Conv → BN → ReLU → Pool (repeat) → Flatten → FC

---
## 📖 Next Steps
➡️ **Next notebook:** [02_cnn_architectures.ipynb](02_cnn_architectures.ipynb) — Famous CNN architectures (LeNet, VGG, ResNet)